In [ ]:
import os
import sys

sys.path.append("../../src/")
import pickle

import lightning as L
import optuna
from lightning import seed_everything
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from optuna.samplers import TPESampler

from model_training.data_modules.utils import EPFDataModule
from model_training.models.mc_dropout_normal import MCDropoutNormal

In [ ]:
import torch

torch.cuda.is_available()

In [ ]:
seed = 0
seed_everything(seed)

In [ ]:
n_trials = 200

In [ ]:
seed = 0
seed_everything(seed)
n_runs = 3
log_dir = "../../logs_hp/"
standardization_case = "mean_std"
forward_passes = 10
name = f"MCDropout{forward_passes}Normal_{standardization_case}_hp"

start_date = None
val_date = "2022-12-01"
test_date = "2023-12-01"
end_date = "2024-11-30"

hidden_dims = [1024, 1024]
max_epochs = 2000

In [ ]:
os.path.exists(f"runs/{name}_s{seed-1}.pkl")

In [ ]:
data_module = EPFDataModule(
    data_file_path="../../data/processed/smard_data_2024-12-11.npz",
    val_date=val_date,
    test_date=test_date,
    end_date=end_date,
    start_date=start_date,
    batch_size=32,
    standardization_case=standardization_case,
)


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    l2_regularization = trial.suggest_float("l2_regularization", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 1e-2, 0.9)

    val_loss = 0
    for i in range(n_runs):
        # Initialize the model
        model = MCDropoutNormal(
            input_dim=data_module.feature_dim,
            output_dim=data_module.target_dim,
            hidden_dims=hidden_dims,
            learning_rate=learning_rate,
            dropout_rate=[dropout_rate],
            l2_regularization=l2_regularization,
            hparams_datamodule=data_module.hparams,
            use_batch_norm=False,
            forward_passes=forward_passes,
            output_postprocessing=(
                data_module.offset_target,
                data_module.scale_target,
            ),
        )

        # Set up the TensorBoard logger
        logger = TensorBoardLogger(
            save_dir=log_dir,
            name=name + f"_s{seed}_n{i}",
            log_graph=False,
            default_hp_metric=False,
        )

        callbacks = []
        early_stop_callback = EarlyStopping(
            monitor="val_loss_early_stopping",
            min_delta=0.0,
            patience=50,
            verbose=False,
            mode="min",
        )
        callbacks.append(early_stop_callback)

        checkpoint_callback = ModelCheckpoint(
            save_top_k=1,
            monitor="val_loss_early_stopping",
            mode="min",
        )
        callbacks.append(checkpoint_callback)

        # Initialize the trainer
        trainer = L.Trainer(
            accelerator="auto",
            devices="auto",
            strategy="auto",
            max_epochs=max_epochs,
            logger=logger,
            deterministic=True,
            log_every_n_steps=0,
            enable_progress_bar=False,
            callbacks=callbacks,
            enable_model_summary=False,
        )

        trainer.fit(model, data_module)

        if "val_loss_neptuna" not in trainer.callback_metrics:
            raise optuna.TrialPruned()
        val_loss += trainer.callback_metrics["val_loss_neptuna"].item()

    print(f"val_loss: {val_loss / n_runs}")
    return val_loss / n_runs

In [ ]:
if os.path.exists(f"runs/{name}_s{seed-1}.pkl"):
    with open(f"runs/{name}_s{seed-1}.pkl", "rb") as f:
        study = pickle.load(f)
    n_trials = n_trials - len(study.trials)
    for _ in range(n_trials):
        study.optimize(objective, n_trials=1)
        with open(f"runs/{name}_s{seed}.pkl", "wb") as f:
            pickle.dump(study, f)

else:
    sampler = TPESampler(seed=seed)

    study = optuna.create_study(
        study_name=name,
        direction="minimize",
        storage=f"sqlite:///{name}.db",
        sampler=sampler,
    )
    for _ in range(n_trials):
        study.optimize(objective, n_trials=1)
        with open(f"runs/{name}_s{seed}.pkl", "wb") as f:
            pickle.dump(study, f)

In [ ]:
print("Finished optimization")
print(study.best_params)
print(study.best_value)
print(study.best_trial)